# ULA Go2 CUDA replay
Choose a GPU in **Runtime → Change runtime type** (T4 is enough). Run the cells in order and upload `ula-gpu-cuda-replay.tar.gz` when prompted. No GitHub, MuJoCo, or training is needed.

Planning, robot dynamics, force problem setup and solve, torque recovery, and motor output run on CUDA. Task calculations still run on the CPU. This checks correctness on recorded inputs, not rollout speed.

In [ ]:
import subprocess as sp
from pathlib import Path

sp.run(['nvidia-smi', '-L'], check=True)
sp.run(['nvcc', '--version'], check=True)
if not Path('/usr/include/eigen3/Eigen/Core').is_file():
    sp.run(['apt-get', '-qq', 'update'], check=True)
    sp.run(['apt-get', '-qq', 'install', '-y', 'libeigen3-dev'], check=True)

In [ ]:
from google.colab import files
from tempfile import mkdtemp
import tarfile

uploaded = files.upload()
assert len(uploaded) == 1, 'Upload only the CUDA replay tar.gz.'
archive = next(iter(uploaded))
run_dir = Path(mkdtemp(prefix='ula-replay-', dir='/content'))
with tarfile.open(archive, 'r:gz') as bundle:
    bundle.extractall(run_dir, filter='data')
repo = run_dir / 'ula-gpu'
assert (repo / 'data/go2_reference.jsonl').is_file(), 'Missing bundled reference trace.'

In [ ]:
print('Building and running the CPU and CUDA replay. The first build can take a few minutes.', flush=True)
result = sp.run(['bash', 'bench/run_cuda_replay.sh'], cwd=repo, text=True, stdout=sp.PIPE, stderr=sp.STDOUT)
log_path = repo / 'cuda_replay.log'
log_path.write_text(result.stdout, encoding='utf-8')
if result.returncode:
    print(result.stdout[-18000:])
else:
    for line in result.stdout.splitlines():
        if any(text in line for text in ('CUDA device ', 'cuda-hybrid:', 'go2_cuda_dynamics ', 'go2_cuda_parity ', 'max |error|', 'motor_torque:', 'tests passed')):
            print(line)
print(f'Full run log: {log_path}')
result.check_returncode()